# Predictive State HMM 01 — volatility-state evidence and separate perp economics

Historical exploratory study. Follow `discussions/Predictive_State_HMM/001_chatgpt_psh01_preregistration.md`. **No live trading and no option PnL without matched Deribit quotes.**

In [1]:
from pathlib import Path
import os, sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/"src"/"data_engineering").exists())
sys.path.insert(0,str(root/"src"))
from dotenv import load_dotenv
load_dotenv(root/".env")
from data_engineering.modeling import predictive_state_hmm as psh
print("Module:",psh.VERSION," Repo:",root)

Module: PSH01-0.1.0  Repo: D:\Projects\deribit_market_data


In [2]:
data_root = Path(os.environ["DERIBIT_DATA_ROOT"]).expanduser()
spot = data_root / "history" / "price_5m"
perps = Path(os.environ.get("BINANCE_UM_DIR",str(data_root.parent/"Binance"))).expanduser()
out = root/"notebooks"/"Predictive_State_HMM"/"01"
print("Spot history:",spot,"\nPerp candles:",perps,"\nOutputs:",out)

Spot history: D:\MarketData\Deribit\history\price_5m 
Perp candles: D:\MarketData\Binance 
Outputs: D:\Projects\deribit_market_data\notebooks\Predictive_State_HMM\01


In [3]:
cfg = psh.Spec()
psh.registration(cfg)

{'version': 'PSH01-0.1.0',
 'assets': ('BTC', 'ETH', 'SOL', 'XRP'),
 'min_history_days': 365,
 'hmm_states': (2, 3, 4),
 'anchor_minutes': 30,
 'primary_shock_horizon_min': 60,
 'perp_horizon_min': 30,
 'perp_roundtrip_cost_bp': 12.0,
 'logistic_C': 0.1,
 'ridge_alpha': 100.0,
 'bootstrap_days': 500,
 'min_test_days': 30,
 'seed': 2617,
 'status': 'HISTORICAL_EXPLORATORY',
 'emissions': ['ema_30m', 'ema_240m'],
 'baseline_features': ['now', 'ema_30m', 'ema_240m', 'clock_sin', 'clock_cos'],
 'allowed_predictor_evidence': {'ema_30m': 'VD12', 'ema_240m': 'VD12'},
 'option_PnL': 'NOT_TESTED_WITHOUT_POINT_IN_TIME_QUOTES',
 'perp_source': 'Binance USD-M historical proxy; not Deribit execution'}

## Model fit and retrospective evaluation

Fits Student-t HMM models K=2/3/4, picks by TRAIN BIC with minimum state occupancy, scores 60-minute jump onset on held-out chronology versus continuous, static-posterior and nonlinear continuous baselines. Independently evaluates 30-minute future returns on the **perpetual contract** using a fixed cost scenario. May take several minutes per asset.

In [4]:
decisions = psh.run(spot,perps,out,cfg)
decisions

[{'asset': 'BTC',
  'K': 4,
  'verdict': 'NO_STABLE_HMM_INCREMENT',
  'vs_static': {'delta_logloss': -0.00038553630768308346,
   'ci025': -0.0007473712053557485,
   'ci975': -4.6704084882613415e-05,
   'test_days': 154},
  'vs_continuous': {'delta_logloss': -0.0003634529063759965,
   'ci025': -0.000710141827362679,
   'ci975': -3.3602733971050856e-05,
   'test_days': 154},
  'vs_nonlinear': {'delta_logloss': 0.0036167643703906662,
   'ci025': 8.429829136270545e-05,
   'ci975': 0.007033217335998879,
   'test_days': 154},
  'prospective_confirmation': False,
  'execution_approved': False},
 {'asset': 'ETH',
  'K': 4,
  'verdict': 'NO_STABLE_HMM_INCREMENT',
  'vs_static': {'delta_logloss': -7.550493094671146e-05,
   'ci025': -0.0005606514353257185,
   'ci975': 0.00039428524353955714,
   'test_days': 154},
  'vs_continuous': {'delta_logloss': -9.806096007308717e-05,
   'ci025': -0.0005893756036971384,
   'ci975': 0.0003833256573946998,
   'test_days': 154},
  'vs_nonlinear': {'delta_loglos

In [5]:
import pandas as pd, json
for filename in ("01_coverage.csv","01_forecast_scores.csv","01_state_profiles.csv","01_duration_hazards.csv"):
    print("\n",filename)
    display(pd.read_csv(out/filename).head(12))
print("\nPerp scenario:")
print(json.dumps(json.loads((out/"01_perp_economics.json").read_text()),indent=2)[:10000])


 01_coverage.csv


,asset,K,bic,min_train_occupancy,n_train,n_valid,n_test
0,BTC,2,94273.907078,0.449927,22271,7440,7393
1,BTC,3,73614.259822,0.287867,22271,7440,7393
2,BTC,4,60599.629513,0.183988,22271,7440,7393
3,ETH,2,95162.429972,0.493574,22271,7440,7393
4,ETH,3,76238.096701,0.285562,22271,7440,7393
5,ETH,4,64491.881455,0.182419,22271,7440,7393
6,SOL,2,94776.949338,0.453885,22271,7440,7393
7,SOL,3,76455.604647,0.259394,22271,7440,7393
8,SOL,4,64278.952841,0.142427,22271,7440,7393
9,XRP,2,89489.555156,0.445409,22271,7440,7393



 01_forecast_scores.csv


,asset,model,K,n_test,n_shock_origins,logloss,brier
0,BTC,continuous,4,7360,379,0.170624,0.044026
1,BTC,static,4,7360,379,0.170646,0.044028
2,BTC,hmm,4,7360,379,0.170260,0.043984
3,BTC,nonlinear,4,7360,379,0.166643,0.043617
4,ETH,continuous,4,7379,306,0.153312,0.037297
5,ETH,static,4,7379,306,0.153289,0.037297
6,ETH,hmm,4,7379,306,0.153214,0.037277
7,ETH,nonlinear,4,7379,306,0.151047,0.036970
8,SOL,continuous,4,7375,298,0.147369,0.035242
9,SOL,static,4,7375,298,0.147339,0.035239



 01_state_profiles.csv


,asset,K,state,occupancy_test,ema30_location_z,ema240_location_z,self_transition
0,BTC,4,0,0.353876,-1.264542,-1.411243,0.971207
1,BTC,4,1,0.229693,0.334379,0.386340,0.956500
2,BTC,4,2,0.337687,-0.342085,-0.372249,0.952786
3,BTC,4,3,0.078744,1.204305,1.348525,0.968397
4,ETH,4,0,0.090997,0.353549,0.430323,0.947884
5,ETH,4,1,0.288731,-0.303230,-0.353110,0.950114
6,ETH,4,2,0.590365,-1.124026,-1.301490,0.969920
7,ETH,4,3,0.029907,1.220492,1.419506,0.965755
8,SOL,4,0,0.100252,0.427517,0.518763,0.950567
9,SOL,4,1,0.677391,-1.008016,-1.142799,0.964715



 01_duration_hazards.csv


,asset,K,state,age_blocks,n_at_risk,n_exits,observed_exit_probability,hmm_geometric_exit_probability
0,BTC,4,0,1,79,9,0.113924,0.028793
1,BTC,4,0,2,70,4,0.057143,0.028793
2,BTC,4,0,3,66,5,0.075758,0.028793
3,BTC,4,0,4,61,1,0.016393,0.028793
4,BTC,4,0,5,60,3,0.050000,0.028793
5,BTC,4,0,6,57,1,0.017544,0.028793
6,BTC,4,0,7,56,1,0.017857,0.028793
7,BTC,4,0,8,55,4,0.072727,0.028793
8,BTC,4,0,9,51,1,0.019608,0.028793
9,BTC,4,0,10,50,3,0.060000,0.028793



Perp scenario:
[
  {
    "asset": "BTC",
    "K": 4,
    "perp_evaluation": {
      "status": "HISTORICAL_PERP_COST_SCENARIO",
      "n_test_origins": 7248,
      "continuous": {
        "n_trades": 0,
        "mean_bp_per_trade_net": null,
        "total_bp_per_notional_slot_net": 0.0,
        "mean_bp_per_origin_net": 0.0,
        "gross_bp_per_trade": null,
        "hit_rate": null
      },
      "static": {
        "n_trades": 0,
        "mean_bp_per_trade_net": null,
        "total_bp_per_notional_slot_net": 0.0,
        "mean_bp_per_origin_net": 0.0,
        "gross_bp_per_trade": null,
        "hit_rate": null
      },
      "hmm": {
        "n_trades": 0,
        "mean_bp_per_trade_net": null,
        "total_bp_per_notional_slot_net": 0.0,
        "mean_bp_per_origin_net": 0.0,
        "gross_bp_per_trade": null,
        "hit_rate": null
      }
    },
    "options_evaluation": "NOT_TESTED_MISSING_MATCHED_DERIBIT_QUOTES"
  },
  {
    "asset": "ETH",
    "K": 4,
    "perp_evalua

## Scientific decision

A positive *historical* result can only authorize a later prospective test. It does not establish real market regimes, an actual executable perpetual edge, or the economics of options. Duration-specific HSMM work remains locked until a model with explicit duration improves new-data forecasts. Push the executed notebook and compact `01/` artifacts for discussion.